In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
def parse_metric_value(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).strip()

    if s == "":
        return np.nan

    try:
        return float(s)
    except Exception:
        pass

    sign = -1 if s.startswith("-") else 1
    s_clean = s.lstrip("+-")

    digits = "".join(ch for ch in s_clean if ch.isdigit())

    if digits == "":
        return np.nan

    if len(digits) == 1:
        return sign * float(digits)

    value = float(digits[0] + "." + digits[1:])

    return sign * value

In [ ]:
import os
import numpy as np
import pandas as pd

from scipy.stats import shapiro, wilcoxon, ttest_rel, rankdata
from statsmodels.stats.multitest import multipletests



# CONFIGURACIÓN GENERAL


METRICS = ["MAE", "RMSE", "R2", "MRE"]

LOWER_IS_BETTER = ["MAE", "RMSE", "MRE"]
HIGHER_IS_BETTER = ["R2"]

STATISTICS_OUTPUT_DIR = "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis"

os.makedirs(STATISTICS_OUTPUT_DIR, exist_ok=True)



# FUNCIONES AUXILIARES


def metric_better_direction(metric):
    if metric in LOWER_IS_BETTER:
        return "lower"

    if metric in HIGHER_IS_BETTER:
        return "higher"

    raise ValueError(f"No se ha definido si la métrica {metric} mejora al subir o al bajar.")


def get_improvement_b_over_a(metric, x_a, x_b):
    """
    Valores positivos significan que strategy_b mejora a strategy_a.
    """

    diff_b_minus_a = x_b - x_a

    if metric_better_direction(metric) == "lower":
        return -diff_b_minus_a

    return diff_b_minus_a


def matched_pairs_rank_biserial_effect(improvement):
    """
    Rank-biserial effect size para datos apareados.

    Positivo: ventaja de strategy_b.
    Negativo: ventaja de strategy_a.
    """

    improvement = np.asarray(improvement, dtype=float)
    improvement = improvement[np.isfinite(improvement)]

    non_zero = improvement[improvement != 0]

    if len(non_zero) == 0:
        return 0.0

    ranks = rankdata(np.abs(non_zero))

    positive_rank_sum = np.sum(ranks[non_zero > 0])
    negative_rank_sum = np.sum(ranks[non_zero < 0])
    total_rank_sum = np.sum(ranks)

    if total_rank_sum == 0:
        return 0.0

    return float((positive_rank_sum - negative_rank_sum) / total_rank_sum)


def safe_shapiro(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 3:
        return np.nan, np.nan

    if np.all(values == values[0]):
        return np.nan, np.nan

    try:
        stat, p = shapiro(values)
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def safe_wilcoxon(x_a, x_b):
    x_a = np.asarray(x_a, dtype=float)
    x_b = np.asarray(x_b, dtype=float)

    mask = np.isfinite(x_a) & np.isfinite(x_b)
    x_a = x_a[mask]
    x_b = x_b[mask]

    diff = x_b - x_a

    if len(diff) < 1:
        return np.nan, np.nan

    if np.all(diff == 0):
        return 0.0, 1.0

    try:
        stat, p = wilcoxon(
            x_b,
            x_a,
            alternative="two-sided",
            zero_method="wilcox"
        )
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def safe_paired_ttest(x_a, x_b):
    x_a = np.asarray(x_a, dtype=float)
    x_b = np.asarray(x_b, dtype=float)

    mask = np.isfinite(x_a) & np.isfinite(x_b)
    x_a = x_a[mask]
    x_b = x_b[mask]

    diff = x_b - x_a

    if len(diff) < 3:
        return np.nan, np.nan

    if np.all(diff == diff[0]):
        return np.nan, np.nan

    try:
        stat, p = ttest_rel(x_b, x_a, alternative="two-sided")
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def check_metric_columns(df, strategy_a, strategy_b, metrics):
    missing = []

    for metric in metrics:
        col_a = f"{metric}_{strategy_a}"
        col_b = f"{metric}_{strategy_b}"

        if col_a not in df.columns:
            missing.append(col_a)

        if col_b not in df.columns:
            missing.append(col_b)

    if missing:
        raise ValueError(
            "Faltan columnas en el TSV comparativo:\n"
            + "\n".join(missing)
        )


# COMPARACIÓN APAREADA GENERAL


def paired_statistical_comparison_general(
    df,
    strategy_a,
    strategy_b,
    comparison_name,
    metrics=("MAE", "RMSE", "R2", "MRE"),
    correction_method="holm",
    alpha=0.05
):
    """
    Comparación apareada entre dos estrategias.

    diff = strategy_b - strategy_a

    improvement_b_over_a:
        positivo significa que strategy_b mejora a strategy_a.

    Para MAE, RMSE y MRE:
        improvement = strategy_a - strategy_b

    Para R2:
        improvement = strategy_b - strategy_a
    """

    check_metric_columns(df, strategy_a, strategy_b, metrics)

    rows = []

    for metric in metrics:
        col_a = f"{metric}_{strategy_a}"
        col_b = f"{metric}_{strategy_b}"

        tmp = df[["id", col_a, col_b]].copy()

        tmp[col_a] = tmp[col_a].apply(parse_metric_value)
        tmp[col_b] = tmp[col_b].apply(parse_metric_value)

        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna()

        x_a = tmp[col_a].to_numpy(dtype=float)
        x_b = tmp[col_b].to_numpy(dtype=float)

        diff_b_minus_a = x_b - x_a
        improvement_b_over_a = get_improvement_b_over_a(metric, x_a, x_b)

        n = len(tmp)

        if n < 3:
            rows.append({
                "comparison": comparison_name,
                "strategy_a": strategy_a,
                "strategy_b": strategy_b,
                "metric": metric,
                "n_pairs": n,
                "mean_strategy_a": np.nan,
                "mean_strategy_b": np.nan,
                "median_strategy_a": np.nan,
                "median_strategy_b": np.nan,
                "mean_diff_b_minus_a": np.nan,
                "median_diff_b_minus_a": np.nan,
                "mean_improvement_b_over_a": np.nan,
                "median_improvement_b_over_a": np.nan,
                "strategy_a_better_count": np.nan,
                "strategy_b_better_count": np.nan,
                "ties_count": np.nan,
                "strategy_a_better_percent": np.nan,
                "strategy_b_better_percent": np.nan,
                "ties_percent": np.nan,
                "rank_biserial_effect_b_over_a": np.nan,
                "shapiro_stat": np.nan,
                "shapiro_p": np.nan,
                "wilcoxon_stat": np.nan,
                "wilcoxon_p": np.nan,
                "ttest_stat": np.nan,
                "ttest_p": np.nan,
                "direction_interpretation": "insufficient_pairs"
            })
            continue

        shapiro_stat, shapiro_p = safe_shapiro(diff_b_minus_a)
        wilcoxon_stat, wilcoxon_p = safe_wilcoxon(x_a, x_b)
        ttest_stat, ttest_p = safe_paired_ttest(x_a, x_b)

        mean_a = float(np.mean(x_a))
        mean_b = float(np.mean(x_b))
        median_a = float(np.median(x_a))
        median_b = float(np.median(x_b))

        mean_diff = float(np.mean(diff_b_minus_a))
        median_diff = float(np.median(diff_b_minus_a))

        mean_improvement = float(np.mean(improvement_b_over_a))
        median_improvement = float(np.median(improvement_b_over_a))

        strategy_b_better_count = int(np.sum(improvement_b_over_a > 0))
        strategy_a_better_count = int(np.sum(improvement_b_over_a < 0))
        ties_count = int(np.sum(improvement_b_over_a == 0))

        strategy_b_better_percent = float(strategy_b_better_count / n * 100)
        strategy_a_better_percent = float(strategy_a_better_count / n * 100)
        ties_percent = float(ties_count / n * 100)

        rank_biserial = matched_pairs_rank_biserial_effect(improvement_b_over_a)

        if median_improvement > 0:
            direction = f"{strategy_b}_better"
        elif median_improvement < 0:
            direction = f"{strategy_a}_better"
        else:
            direction = "no_direction"

        rows.append({
            "comparison": comparison_name,
            "strategy_a": strategy_a,
            "strategy_b": strategy_b,
            "metric": metric,
            "n_pairs": n,

            "mean_strategy_a": mean_a,
            "mean_strategy_b": mean_b,
            "median_strategy_a": median_a,
            "median_strategy_b": median_b,

            "mean_diff_b_minus_a": mean_diff,
            "median_diff_b_minus_a": median_diff,

            "mean_improvement_b_over_a": mean_improvement,
            "median_improvement_b_over_a": median_improvement,

            "strategy_a_better_count": strategy_a_better_count,
            "strategy_b_better_count": strategy_b_better_count,
            "ties_count": ties_count,

            "strategy_a_better_percent": strategy_a_better_percent,
            "strategy_b_better_percent": strategy_b_better_percent,
            "ties_percent": ties_percent,

            "rank_biserial_effect_b_over_a": rank_biserial,

            "shapiro_stat": shapiro_stat,
            "shapiro_p": shapiro_p,

            "wilcoxon_stat": wilcoxon_stat,
            "wilcoxon_p": wilcoxon_p,

            "ttest_stat": ttest_stat,
            "ttest_p": ttest_p,

            "direction_interpretation": direction
        })

    results_df = pd.DataFrame(rows)

    valid_wilcoxon = results_df["wilcoxon_p"].notna()
    results_df["wilcoxon_p_corrected_within_comparison"] = np.nan
    results_df["wilcoxon_reject_H0_within_comparison"] = False

    if valid_wilcoxon.sum() > 0:
        reject, p_corrected, _, _ = multipletests(
            results_df.loc[valid_wilcoxon, "wilcoxon_p"],
            alpha=alpha,
            method=correction_method
        )

        results_df.loc[valid_wilcoxon, "wilcoxon_p_corrected_within_comparison"] = p_corrected
        results_df.loc[valid_wilcoxon, "wilcoxon_reject_H0_within_comparison"] = reject

    valid_ttest = results_df["ttest_p"].notna()
    results_df["ttest_p_corrected_within_comparison"] = np.nan
    results_df["ttest_reject_H0_within_comparison"] = False

    if valid_ttest.sum() > 0:
        reject_t, p_corrected_t, _, _ = multipletests(
            results_df.loc[valid_ttest, "ttest_p"],
            alpha=alpha,
            method=correction_method
        )

        results_df.loc[valid_ttest, "ttest_p_corrected_within_comparison"] = p_corrected_t
        results_df.loc[valid_ttest, "ttest_reject_H0_within_comparison"] = reject_t

    return results_df



# EJECUTAR UNA SOLA COMPARACIÓN


def run_single_pairwise_statistical_test(
    pairwise_tsv_path,
    strategy_a,
    strategy_b,
    comparison_name,
    output_tsv_path,
    metrics=("MAE", "RMSE", "R2", "MRE"),
    correction_method="holm",
    alpha=0.05
):
    if not os.path.exists(pairwise_tsv_path):
        raise FileNotFoundError(f"No existe el archivo: {pairwise_tsv_path}")

    df = pd.read_csv(pairwise_tsv_path, sep="\t")

    result_df = paired_statistical_comparison_general(
        df=df,
        strategy_a=strategy_a,
        strategy_b=strategy_b,
        comparison_name=comparison_name,
        metrics=metrics,
        correction_method=correction_method,
        alpha=alpha
    )

    output_dir = os.path.dirname(output_tsv_path)

    if output_dir != "":
        os.makedirs(output_dir, exist_ok=True)

    result_df.to_csv(output_tsv_path, sep="\t", index=False)

    print("Análisis completado.")
    print(f"Comparación: {comparison_name}")
    print(f"Estrategia A: {strategy_a}")
    print(f"Estrategia B: {strategy_b}")
    print(f"Guardado en: {output_tsv_path}")

    return result_df

In [ ]:
pairwise_path = "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_fixed_vs_global_fixed.tsv"

output_path = "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_global_fixed.tsv"

stats_individual_fixed_vs_global_fixed = run_single_pairwise_statistical_test(
    pairwise_tsv_path=pairwise_path,
    strategy_a="individual_fixed",
    strategy_b="global_fixed",
    comparison_name="individual_fixed_vs_global_fixed",
    output_tsv_path=output_path,
    metrics=METRICS,
    correction_method="holm",
    alpha=0.05
)

display(stats_individual_fixed_vs_global_fixed)

Análisis completado.
Comparación: individual_fixed_vs_global_fixed
Estrategia A: individual_fixed
Estrategia B: global_fixed
Guardado en: /content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_global_fixed.tsv


,comparison,strategy_a,strategy_b,metric,n_pairs,mean_strategy_a,mean_strategy_b,median_strategy_a,median_strategy_b,mean_diff_b_minus_a,median_diff_b_minus_a,mean_improvement_b_over_a,median_improvement_b_over_a,strategy_a_better_count,strategy_b_better_count,ties_count,strategy_a_better_percent,strategy_b_better_percent,ties_percent,rank_biserial_effect_b_over_a,shapiro_stat,shapiro_p,wilcoxon_stat,wilcoxon_p,ttest_stat,ttest_p,direction_interpretation,wilcoxon_p_corrected_within_comparison,wilcoxon_reject_H0_within_comparison,ttest_p_corrected_within_comparison,ttest_reject_H0_within_comparison
0,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,MAE,141,1.023771,0.736303,0.884704,0.513284,-0.287468,-0.296794,0.287468,0.296794,26,115,0,18.439716,81.560284,0.0,0.681950,0.944367,2.058820e-05,1592.0,2.137626e-12,-7.423805,1.009507e-11,global_fixed_better,2.137626e-12,True,1.009507e-11,True
1,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,RMSE,141,1.604135,1.051528,1.333342,0.688894,-0.552607,-0.586309,0.552607,0.586309,21,120,0,14.893617,85.106383,0.0,0.813805,0.888697,7.231015e-09,932.0,5.136802e-17,-10.342778,5.615177e-19,global_fixed_better,1.027360e-16,True,1.123035e-18,True
2,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,R2,141,0.630029,0.833361,0.666636,0.900506,0.203333,0.193760,0.203333,0.193760,21,120,0,14.893617,85.106383,0.0,0.836979,0.978941,2.835245e-02,816.0,6.564824e-18,11.345255,1.456183e-21,global_fixed_better,1.969447e-17,True,4.368550e-21,True
3,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,MRE,141,0.581661,0.269424,0.552025,0.193300,-0.312237,-0.343228,0.312237,0.343228,20,121,0,14.184397,85.815603,0.0,0.862351,0.980927,4.631200e-02,689.0,6.471378e-19,-12.692070,4.804558e-25,global_fixed_better,2.588551e-18,True,1.921823e-24,True


In [ ]:
pairwise_path = "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_fixed_vs_individual_optuna.tsv"

output_path = "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_individual_optuna.tsv"

stats_individual_fixed_vs_individual_optuna = run_single_pairwise_statistical_test(
    pairwise_tsv_path=pairwise_path,
    strategy_a="individual_fixed",
    strategy_b="individual_optuna",
    comparison_name="individual_fixed_vs_individual_optuna",
    output_tsv_path=output_path,
    metrics=METRICS,
    correction_method="holm",
    alpha=0.05
)

display(stats_individual_fixed_vs_individual_optuna)

Análisis completado.
Comparación: individual_fixed_vs_individual_optuna
Estrategia A: individual_fixed
Estrategia B: individual_optuna
Guardado en: /content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_individual_optuna.tsv


,comparison,strategy_a,strategy_b,metric,n_pairs,mean_strategy_a,mean_strategy_b,median_strategy_a,median_strategy_b,mean_diff_b_minus_a,median_diff_b_minus_a,mean_improvement_b_over_a,median_improvement_b_over_a,strategy_a_better_count,strategy_b_better_count,ties_count,strategy_a_better_percent,strategy_b_better_percent,ties_percent,rank_biserial_effect_b_over_a,shapiro_stat,shapiro_p,wilcoxon_stat,wilcoxon_p,ttest_stat,ttest_p,direction_interpretation,wilcoxon_p_corrected_within_comparison,wilcoxon_reject_H0_within_comparison,ttest_p_corrected_within_comparison,ttest_reject_H0_within_comparison
0,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,MAE,141,1.023771,0.772229,0.884704,0.605180,-0.251542,-0.249140,0.251542,0.249140,10,131,0,7.092199,92.907801,0.0,0.974029,0.912809,1.559534e-07,130.0,1.079541e-23,-16.061502,1.445087e-33,individual_optuna_better,1.307382e-23,True,4.335260e-33,True
1,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,RMSE,141,1.604135,1.216535,1.333342,0.882925,-0.387600,-0.376017,0.387600,0.376017,6,135,0,4.255319,95.744681,0.0,0.985816,0.961766,5.704396e-04,71.0,3.131618e-24,-19.092914,8.391095e-41,individual_optuna_better,9.394853e-24,True,3.356438e-40,True
2,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,R2,141,0.630029,0.824024,0.666636,0.841180,0.193995,0.141501,0.193995,0.141501,6,135,0,4.255319,95.744681,0.0,0.978823,0.373395,5.205680e-22,106.0,6.536912e-24,6.713603,4.373374e-10,individual_optuna_better,1.307382e-23,True,4.373374e-10,True
3,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,MRE,141,0.581661,0.324566,0.552025,0.309725,-0.257095,-0.234333,0.257095,0.234333,7,134,0,4.964539,95.035461,0.0,0.989412,0.901227,3.370281e-08,53.0,2.140581e-24,-14.948946,8.470497e-31,individual_optuna_better,8.562324e-24,True,1.694099e-30,True


In [ ]:
pairwise_path = "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_global_fixed_vs_global_optuna.tsv"

output_path = "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_global_fixed_vs_global_optuna.tsv"

stats_global_fixed_vs_global_optuna = run_single_pairwise_statistical_test(
    pairwise_tsv_path=pairwise_path,
    strategy_a="global_fixed",
    strategy_b="global_optuna",
    comparison_name="global_fixed_vs_global_optuna",
    output_tsv_path=output_path,
    metrics=METRICS,
    correction_method="holm",
    alpha=0.05
)

display(stats_global_fixed_vs_global_optuna)

Análisis completado.
Comparación: global_fixed_vs_global_optuna
Estrategia A: global_fixed
Estrategia B: global_optuna
Guardado en: /content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_global_fixed_vs_global_optuna.tsv


,comparison,strategy_a,strategy_b,metric,n_pairs,mean_strategy_a,mean_strategy_b,median_strategy_a,median_strategy_b,mean_diff_b_minus_a,median_diff_b_minus_a,mean_improvement_b_over_a,median_improvement_b_over_a,strategy_a_better_count,strategy_b_better_count,ties_count,strategy_a_better_percent,strategy_b_better_percent,ties_percent,rank_biserial_effect_b_over_a,shapiro_stat,shapiro_p,wilcoxon_stat,wilcoxon_p,ttest_stat,ttest_p,direction_interpretation,wilcoxon_p_corrected_within_comparison,wilcoxon_reject_H0_within_comparison,ttest_p_corrected_within_comparison,ttest_reject_H0_within_comparison
0,global_fixed_vs_global_optuna,global_fixed,global_optuna,MAE,141,0.736303,0.533551,0.513284,0.397657,-0.202752,-0.111974,0.202752,0.111974,16,125,0,11.347518,88.652482,0.0,0.900110,0.775595,2.048336e-13,500.0,1.816574e-20,-8.919180,2.315752e-15,global_optuna_better,5.449723e-20,True,9.263010e-15,True
1,global_fixed_vs_global_optuna,global_fixed,global_optuna,RMSE,141,1.051528,0.830295,0.688894,0.569164,-0.221233,-0.119669,0.221233,0.119669,19,122,0,13.475177,86.524823,0.0,0.838178,0.783578,3.736753e-13,810.0,5.893108e-18,-7.339857,1.589369e-11,global_optuna_better,1.178622e-17,True,3.178738e-11,True
2,global_fixed_vs_global_optuna,global_fixed,global_optuna,R2,141,0.833361,0.906980,0.900506,0.938873,0.073619,0.020700,0.073619,0.020700,19,122,0,13.475177,86.524823,0.0,0.806613,0.314762,7.217563e-23,968.0,9.616323e-17,3.623616,4.055586e-04,global_optuna_better,9.616323e-17,True,4.055586e-04,True
3,global_fixed_vs_global_optuna,global_fixed,global_optuna,MRE,141,0.269424,0.169366,0.193300,0.126474,-0.100058,-0.062428,0.100058,0.062428,10,131,0,7.092199,92.907801,0.0,0.977625,0.624052,1.909391e-17,112.0,7.412039e-24,-8.676854,9.296874e-15,global_optuna_better,2.964816e-23,True,2.789062e-14,True


In [ ]:
pairwise_path = "/content/drive/MyDrive/TFG_Optuna/Pairwise_Comparisons/pairwise_individual_optuna_vs_global_optuna.tsv"

output_path = "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_optuna_vs_global_optuna.tsv"

stats_individual_optuna_vs_global_optuna = run_single_pairwise_statistical_test(
    pairwise_tsv_path=pairwise_path,
    strategy_a="individual_optuna",
    strategy_b="global_optuna",
    comparison_name="individual_optuna_vs_global_optuna",
    output_tsv_path=output_path,
    metrics=METRICS,
    correction_method="holm",
    alpha=0.05
)

display(stats_individual_optuna_vs_global_optuna)

Análisis completado.
Comparación: individual_optuna_vs_global_optuna
Estrategia A: individual_optuna
Estrategia B: global_optuna
Guardado en: /content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_optuna_vs_global_optuna.tsv


,comparison,strategy_a,strategy_b,metric,n_pairs,mean_strategy_a,mean_strategy_b,median_strategy_a,median_strategy_b,mean_diff_b_minus_a,median_diff_b_minus_a,mean_improvement_b_over_a,median_improvement_b_over_a,strategy_a_better_count,strategy_b_better_count,ties_count,strategy_a_better_percent,strategy_b_better_percent,ties_percent,rank_biserial_effect_b_over_a,shapiro_stat,shapiro_p,wilcoxon_stat,wilcoxon_p,ttest_stat,ttest_p,direction_interpretation,wilcoxon_p_corrected_within_comparison,wilcoxon_reject_H0_within_comparison,ttest_p_corrected_within_comparison,ttest_reject_H0_within_comparison
0,individual_optuna_vs_global_optuna,individual_optuna,global_optuna,MAE,141,0.772229,0.533551,0.605180,0.397657,-0.238678,-0.236429,0.238678,0.236429,27,114,0,19.148936,80.851064,0.0,0.799221,0.823825,1.003338e-11,1005.0,1.821539e-16,-7.885901,8.010556e-13,global_optuna_better,1.821539e-16,True,9.362567e-13,True
1,individual_optuna_vs_global_optuna,individual_optuna,global_optuna,RMSE,141,1.216535,0.830295,0.882925,0.569164,-0.386240,-0.341272,0.386240,0.341272,14,127,0,9.929078,90.070922,0.0,0.897912,0.690786,7.511995e-16,511.0,2.245675e-20,-7.982580,4.681284e-13,global_optuna_better,8.982702e-20,True,9.362567e-13,True
2,individual_optuna_vs_global_optuna,individual_optuna,global_optuna,R2,141,0.824024,0.906980,0.841180,0.938873,0.082956,0.069161,0.082956,0.069161,14,127,0,9.929078,90.070922,0.0,0.862951,0.724346,5.948533e-15,686.0,6.121717e-19,8.240308,1.105985e-13,global_optuna_better,1.224343e-18,True,3.317956e-13,True
3,individual_optuna_vs_global_optuna,individual_optuna,global_optuna,MRE,141,0.324566,0.169366,0.309725,0.126474,-0.155200,-0.167227,0.155200,0.167227,14,127,0,9.929078,90.070922,0.0,0.888123,0.957920,2.603848e-04,560.0,5.739959e-20,-12.722010,4.022154e-25,global_optuna_better,1.721988e-19,True,1.608862e-24,True


In [ ]:
def combine_saved_statistical_tests(
    saved_statistical_tsv_paths,
    output_tsv_path,
    correction_method="holm",
    alpha=0.05
):
    dfs = []

    for path in saved_statistical_tsv_paths:
        if not os.path.exists(path):
            print(f"No existe y se omite: {path}")
            continue

        df = pd.read_csv(path, sep="\t")
        dfs.append(df)

    if len(dfs) == 0:
        raise ValueError("No se encontró ningún TSV de análisis estadístico.")

    combined_df = pd.concat(dfs, ignore_index=True)

    valid_wilcoxon = combined_df["wilcoxon_p"].notna()
    combined_df["wilcoxon_p_corrected_global"] = np.nan
    combined_df["wilcoxon_reject_H0_global"] = False

    if valid_wilcoxon.sum() > 0:
        reject, p_corrected, _, _ = multipletests(
            combined_df.loc[valid_wilcoxon, "wilcoxon_p"],
            alpha=alpha,
            method=correction_method
        )

        combined_df.loc[valid_wilcoxon, "wilcoxon_p_corrected_global"] = p_corrected
        combined_df.loc[valid_wilcoxon, "wilcoxon_reject_H0_global"] = reject

    valid_ttest = combined_df["ttest_p"].notna()
    combined_df["ttest_p_corrected_global"] = np.nan
    combined_df["ttest_reject_H0_global"] = False

    if valid_ttest.sum() > 0:
        reject_t, p_corrected_t, _, _ = multipletests(
            combined_df.loc[valid_ttest, "ttest_p"],
            alpha=alpha,
            method=correction_method
        )

        combined_df.loc[valid_ttest, "ttest_p_corrected_global"] = p_corrected_t
        combined_df.loc[valid_ttest, "ttest_reject_H0_global"] = reject_t

    output_dir = os.path.dirname(output_tsv_path)

    if output_dir != "":
        os.makedirs(output_dir, exist_ok=True)

    combined_df.to_csv(output_tsv_path, sep="\t", index=False)

    print(f"Resumen global guardado en: {output_tsv_path}")

    return combined_df

In [ ]:
saved_paths = [
    "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_global_fixed.tsv",
    "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_fixed_vs_individual_optuna.tsv",
    "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_global_fixed_vs_global_optuna.tsv",
    "/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_individual_optuna_vs_global_optuna.tsv",
]

combined_stats = combine_saved_statistical_tests(
    saved_statistical_tsv_paths=saved_paths,
    output_tsv_path="/content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_combined_available.tsv",
    correction_method="holm",
    alpha=0.05
)

display(combined_stats)

Resumen global guardado en: /content/drive/MyDrive/TFG_Optuna/Statistical_Analysis/statistical_tests_combined_available.tsv


,comparison,strategy_a,strategy_b,metric,n_pairs,mean_strategy_a,mean_strategy_b,median_strategy_a,median_strategy_b,mean_diff_b_minus_a,median_diff_b_minus_a,mean_improvement_b_over_a,median_improvement_b_over_a,strategy_a_better_count,strategy_b_better_count,ties_count,strategy_a_better_percent,strategy_b_better_percent,ties_percent,rank_biserial_effect_b_over_a,shapiro_stat,shapiro_p,wilcoxon_stat,wilcoxon_p,ttest_stat,ttest_p,direction_interpretation,wilcoxon_p_corrected_within_comparison,wilcoxon_reject_H0_within_comparison,ttest_p_corrected_within_comparison,ttest_reject_H0_within_comparison,wilcoxon_p_corrected_global,wilcoxon_reject_H0_global,ttest_p_corrected_global,ttest_reject_H0_global
0,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,MAE,141,1.023771,0.736303,0.884704,0.513284,-0.287468,-0.296794,0.287468,0.296794,26,115,0,18.439716,81.560284,0.0,0.681950,0.944367,2.058820e-05,1592.0,2.137626e-12,-7.423805,1.009507e-11,global_fixed_better,2.137626e-12,True,1.009507e-11,True,2.137626e-12,True,4.038030e-11,True
1,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,RMSE,141,1.604135,1.051528,1.333342,0.688894,-0.552607,-0.586309,0.552607,0.586309,21,120,0,14.893617,85.106383,0.0,0.813805,0.888697,7.231015e-09,932.0,5.136802e-17,-10.342778,5.615177e-19,global_fixed_better,1.027360e-16,True,1.123035e-18,True,2.054721e-16,True,5.615177e-18,True
2,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,R2,141,0.630029,0.833361,0.666636,0.900506,0.203333,0.193760,0.203333,0.193760,21,120,0,14.893617,85.106383,0.0,0.836979,0.978941,2.835245e-02,816.0,6.564824e-18,11.345255,1.456183e-21,global_fixed_better,1.969447e-17,True,4.368550e-21,True,3.535865e-17,True,1.601802e-20,True
3,individual_fixed_vs_global_fixed,individual_fixed,global_fixed,MRE,141,0.581661,0.269424,0.552025,0.193300,-0.312237,-0.343228,0.312237,0.343228,20,121,0,14.184397,85.815603,0.0,0.862351,0.980927,4.631200e-02,689.0,6.471378e-19,-12.692070,4.804558e-25,global_fixed_better,2.588551e-18,True,1.921823e-24,True,4.897373e-18,True,5.765470e-24,True
4,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,MAE,141,1.023771,0.772229,0.884704,0.605180,-0.251542,-0.249140,0.251542,0.249140,10,131,0,7.092199,92.907801,0.0,0.974029,0.912809,1.559534e-07,130.0,1.079541e-23,-16.061502,1.445087e-33,individual_optuna_better,1.307382e-23,True,4.335260e-33,True,1.295450e-22,True,2.167630e-32,True
5,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,RMSE,141,1.604135,1.216535,1.333342,0.882925,-0.387600,-0.376017,0.387600,0.376017,6,135,0,4.255319,95.744681,0.0,0.985816,0.961766,5.704396e-04,71.0,3.131618e-24,-19.092914,8.391095e-41,individual_optuna_better,9.394853e-24,True,3.356438e-40,True,4.697426e-23,True,1.342575e-39,True
6,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,R2,141,0.630029,0.824024,0.666636,0.841180,0.193995,0.141501,0.193995,0.141501,6,135,0,4.255319,95.744681,0.0,0.978823,0.373395,5.205680e-22,106.0,6.536912e-24,6.713603,4.373374e-10,individual_optuna_better,1.307382e-23,True,4.373374e-10,True,9.151676e-23,True,8.746747e-10,True
7,individual_fixed_vs_individual_optuna,individual_fixed,individual_optuna,MRE,141,0.581661,0.324566,0.552025,0.309725,-0.257095,-0.234333,0.257095,0.234333,7,134,0,4.964539,95.035461,0.0,0.989412,0.901227,3.370281e-08,53.0,2.140581e-24,-14.948946,8.470497e-31,individual_optuna_better,8.562324e-24,True,1.694099e-30,True,3.424929e-23,True,1.185870e-29,True
8,global_fixed_vs_global_optuna,global_fixed,global_optuna,MAE,141,0.736303,0.533551,0.513284,0.397657,-0.202752,-0.111974,0.202752,0.111974,16,125,0,11.347518,88.652482,0.0,0.900110,0.775595,2.048336e-13,500.0,1.816574e-20,-8.919180,2.315752e-15,global_optuna_better,5.449723e-20,True,9.263010e-15,True,1.998232e-19,True,2.084177e-14,True
9,global_fixed_vs_global_optuna,global_fixed,global_optuna,RMSE,141,1.051528,0.830295,0.688894,0.569164,-0.221233,-0.119669,0.221233,0.119669